# 01 — Exploration des données brutes (Raw)
## Analyse du fichier `wti_history_init.parquet`

Ce notebook charge le fichier Parquet produit par le script d'ingestion Yahoo Finance et effectue une **analyse complète** :

1. **Chargement** et aperçu du DataFrame
2. **Statistiques descriptives** (min, max, moyenne, écart-type…)
3. **Vérification de la qualité** (valeurs manquantes, doublons)
4. **Analyse temporelle** (volume de données par jour/heure)
5. **Visualisations** (prix de clôture, chandeliers, volumes, distribution)

In [ ]:
# ═══════════════════════════════════════════════
# 📦 Imports
# ═══════════════════════════════════════════════
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Style des graphiques
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# Chemin du fichier Parquet
PARQUET_PATH = os.path.join("..", "raw", "yahoofinance", "history", "wti_history_init.parquet")
print(f"Fichier ciblé : {os.path.abspath(PARQUET_PATH)}")

## 1. Chargement et aperçu du DataFrame

In [ ]:
# ═══════════════════════════════════════════════
# 1. Chargement du fichier Parquet
# ═══════════════════════════════════════════════
df = pd.read_parquet(PARQUET_PATH, engine="pyarrow")

print(f"Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(df.dtypes)
print(f"\n{'═' * 60}")
print("Aperçu des 5 premières lignes :")
df.head()

In [ ]:
# Dernières lignes
print("Aperçu des 5 dernières lignes :")
df.tail()

## 2. Statistiques descriptives

In [ ]:
# ═══════════════════════════════════════════════
# 2. Statistiques descriptives complètes
# ═══════════════════════════════════════════════
print("Statistiques descriptives :\n")
df.describe().round(2)

In [ ]:
# ═══════════════════════════════════════════════
# Informations sur la plage temporelle
# ═══════════════════════════════════════════════
print(f"Plage temporelle : {df['Datetime'].min()} → {df['Datetime'].max()}")
print(f"Durée totale     : {df['Datetime'].max() - df['Datetime'].min()}")
print(f"Nombre de jours uniques : {df['Datetime'].dt.date.nunique()}")

# Prix WTI — résumé rapide
print(f"\n{'═' * 60}")
print(f"Prix WTI (Close) :")
print(f"  Min      : {df['Close'].min():.2f} $")
print(f"  Max      : {df['Close'].max():.2f} $")
print(f"  Moyenne  : {df['Close'].mean():.2f} $")
print(f"  Médiane  : {df['Close'].median():.2f} $")
print(f"  Écart-type : {df['Close'].std():.2f} $")
print(f"  Amplitude  : {df['Close'].max() - df['Close'].min():.2f} $")

## 3. Qualité des données (valeurs manquantes, doublons)

In [ ]:
# ═══════════════════════════════════════════════
# 3. Qualité des données
# ═══════════════════════════════════════════════

# 3a. Valeurs manquantes par colonne
print("Valeurs manquantes (NaN) par colonne :")
missing = df.isnull().sum()
print(missing.to_string())
print(f"\nTotal NaN : {missing.sum()} / {df.shape[0] * df.shape[1]} cellules ({missing.sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%)")

# 3b. Lignes entièrement dupliquées
doublons = df.duplicated().sum()
print(f"\n{'═' * 60}")
print(f"Lignes dupliquées : {doublons}")

# 3c. Doublons sur Datetime uniquement (même timestamp)
doublons_datetime = df.duplicated(subset=["Datetime"]).sum()
print(f"Timestamps en double : {doublons_datetime}")

# 3d. Vérification de la continuité temporelle
print(f"\n{'═' * 60}")
print("Continuité temporelle :")
df_sorted = df.sort_values("Datetime")
time_diffs = df_sorted["Datetime"].diff().dropna()
print(f"  Intervalle le plus fréquent : {time_diffs.mode().iloc[0]}")
print(f"  Intervalle min              : {time_diffs.min()}")
print(f"  Intervalle max              : {time_diffs.max()} (gaps = fermeture marché)")

## 4. Analyse temporelle (volume de données par jour / heure)

In [ ]:
# ═══════════════════════════════════════════════
# 4. Analyse temporelle — Nombre de bougies par jour
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 4a. Nombre de lignes par jour
daily_counts = df.set_index("Datetime").resample("D").size()
daily_counts = daily_counts[daily_counts > 0]  # Exclure les jours sans données

axes[0].bar(daily_counts.index, daily_counts.values, color="steelblue", alpha=0.8)
axes[0].set_title("Nombre de bougies (15min) par jour", fontweight="bold")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Nombre de bougies")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
axes[0].tick_params(axis="x", rotation=45)

# 4b. Distribution par heure de la journée (UTC)
df["Heure"] = df["Datetime"].dt.hour
hourly_counts = df.groupby("Heure").size()

axes[1].bar(hourly_counts.index, hourly_counts.values, color="coral", alpha=0.8)
axes[1].set_title("Distribution des données par heure (UTC)", fontweight="bold")
axes[1].set_xlabel("Heure (UTC)")
axes[1].set_ylabel("Nombre de bougies")
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

print(f"Jours de trading dans la période : {len(daily_counts)}")
print(f"Moyenne de bougies par jour : {daily_counts.mean():.0f}")

## 5. Visualisations

### 5a. Évolution du prix de clôture (Close)

In [ ]:
# ═══════════════════════════════════════════════
# 5a. Évolution du prix de clôture WTI
# ═══════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df["Datetime"], df["Close"], linewidth=0.6, color="royalblue", alpha=0.9)
ax.fill_between(df["Datetime"], df["Close"], alpha=0.15, color="royalblue")

# Moyenne mobile 24h (96 bougies de 15min)
ma_24h = df["Close"].rolling(window=96, min_periods=1).mean()
ax.plot(df["Datetime"], ma_24h, linewidth=1.5, color="orange", label="Moyenne mobile 24h")

ax.set_title("Prix de clôture WTI — Intervalle 15 minutes", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Prix (USD)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5b. Graphique en chandeliers (Candlestick) — Derniers 5 jours

In [ ]:
# ═══════════════════════════════════════════════
# 5b. Chandeliers japonais (derniers 5 jours)
# ═══════════════════════════════════════════════
from matplotlib.patches import FancyBboxPatch

# Filtrer les 5 derniers jours de trading
last_dates = sorted(df["Datetime"].dt.date.unique())[-5:]
df_last = df[df["Datetime"].dt.date.isin(last_dates)].copy()

fig, ax = plt.subplots(figsize=(16, 6))

width = 0.005  # Largeur des bougies
for idx, row in df_last.iterrows():
    color = "green" if row["Close"] >= row["Open"] else "red"
    # Mèche (High - Low)
    ax.plot([row["Datetime"], row["Datetime"]], [row["Low"], row["High"]],
            color=color, linewidth=0.6)
    # Corps (Open - Close)
    ax.bar(row["Datetime"], row["Close"] - row["Open"], bottom=row["Open"],
           width=width, color=color, alpha=0.8, edgecolor=color)

ax.set_title(f"Chandeliers WTI 15min — {last_dates[0]} → {last_dates[-1]}", fontsize=14, fontweight="bold")
ax.set_xlabel("Date / Heure (UTC)")
ax.set_ylabel("Prix (USD)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m %H:%M"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()